# BRIDGE-JUPYTER-01 — PostgreSQL magics in Jupyter

## Goal

Use JupySQL to explore the disposable course PostgreSQL database from a notebook without embedding or displaying credentials. Keep SQL structure separate from data values, bound result size, and know when an application should use Psycopg directly instead.

**Level:** Intermediate/advanced  
**Stable lesson ID:** `bridge-jupyter-01`  
**Prerequisites:** Python Day 18, SQL Day 15, Bridge Day 3, and a reset disposable course database.

An IPython **line magic** begins with one `%` and consumes the rest of that line. A **cell magic** begins with `%%` and consumes the cell body. Here `%sql` is convenient for a short statement or a result assignment; `%%sql` keeps a multi-line query readable.

## Setup

Complete the repository setup outside this notebook. Start VS Code or Jupyter from a shell where `DS60_DATABASE_URL` is set to the disposable `advanced_sql_training` database. Do not paste that value into a cell, print it, or save it in notebook output. The companion guide has separate Windows PowerShell and macOS/Linux commands.

In [ ]:
# ruff: noqa: E501 -- JupySQL line magics stay on one physical line.
%load_ext sql

JupySQL accepts a SQLAlchemy engine. The next cell parses the environment value, confirms the safe course target, selects SQLAlchemy's Psycopg 3 dialect, and creates a lazy engine. `create_engine()` does not print the URL and normally does not connect until first use.

In [ ]:
import os

from sqlalchemy import create_engine
from sqlalchemy.engine import make_url

raw_database_url = os.environ.get("DS60_DATABASE_URL", "").strip()
if not raw_database_url:
    raise RuntimeError(
        "Set DS60_DATABASE_URL in the shell that starts this notebook, then restart the kernel."
    )

course_url = make_url(raw_database_url)
if course_url.get_backend_name() not in {"postgres", "postgresql"}:
    raise RuntimeError("DS60_DATABASE_URL must select PostgreSQL.")
if course_url.database != "advanced_sql_training":
    raise RuntimeError("This lesson is restricted to the disposable course database.")

psycopg_url = course_url.set(drivername="postgresql+psycopg")
engine = create_engine(psycopg_url, pool_pre_ping=True)
assert engine.url.drivername == "postgresql+psycopg"

Hide connection feedback before binding the engine. `autolimit` bounds rows fetched; `displaylimit` only shortens what is displayed and does not by itself protect memory. Keep `autopandas=False` initially so you can practise explicit conversion. Named `:value` parameters are enabled for safe value binding.

In [ ]:
%config SqlMagic.displaycon = False
%config SqlMagic.autolimit = 200
%config SqlMagic.displaylimit = 25
%config SqlMagic.autopandas = False
%config SqlMagic.named_parameters = "enabled"

In [ ]:
%sql engine --alias ds60-course

In [ ]:
%sql --connections

## Steps

### 1. Compare line and cell magics

Use a line magic for a small diagnostic. Use a cell magic when SQL should span multiple lines. These cells are read-only.

In [ ]:
%sql SELECT current_database() AS database_name, current_user AS database_user

In [ ]:
%%sql
SELECT customer_id, full_name, country, segment
FROM training.customers
ORDER BY customer_id
LIMIT 5;

### 2. Bind data values and convert a result

Python variables referenced as `:value` are sent through the database parameter boundary. They are data, not executable SQL. The query text keeps static table and column names.

In [ ]:
order_status = "paid"
minimum_total = 250

In [ ]:
orders_result = %sql SELECT order_id, customer_id, total_amount FROM training.orders WHERE status = :order_status AND total_amount >= :minimum_total ORDER BY total_amount DESC, order_id LIMIT 20

In [ ]:
orders_frame = orders_result.DataFrame()
assert list(orders_frame.columns) == ["order_id", "customer_id", "total_amount"]
orders_frame.head()

### 3. Use `autopandas` deliberately

With `autopandas=True`, an assigned `%sql` result is already a pandas DataFrame. Pandas controls display in this mode, so keep an explicit SQL `LIMIT` as well as `autolimit`.

In [ ]:
%config SqlMagic.autopandas = True
customer_frame = %sql SELECT customer_id, full_name, country FROM training.customers ORDER BY customer_id LIMIT 10
%config SqlMagic.autopandas = False

In [ ]:
assert customer_frame.shape[0] <= 10
assert {"customer_id", "full_name", "country"}.issubset(customer_frame.columns)

### 4. Separate binding from code generation

JupySQL also supports Jinja text such as `{{value}}`. That renders SQL source *before* the driver executes it, so an untrusted value can change SQL structure. Treat Jinja as reviewed code generation, not value binding. Prefer `:value` for data.

A parameter cannot represent an identifier: `FROM :table_name` is not valid identifier binding. Keep identifiers static in exploratory notebooks. In application code, validate a genuinely dynamic choice with an allowlist and compose it with `psycopg.sql.Identifier`.

In [ ]:
chosen_customer_id = 3
customer_orders = %sql SELECT order_id, order_date, status, total_amount FROM training.orders WHERE customer_id = :chosen_customer_id ORDER BY order_date, order_id LIMIT 25

### 5. Treat transactions as an explicit boundary

JupySQL's `SqlMagic.autocommit` defaults to true. A notebook is easy to run out of order, so this lesson stays read-only. For a multi-statement write, use a reviewed `engine.begin()` block or Psycopg transaction context and make commit/rollback ownership visible. Do not assume toggling a magic setting gives an application a complete retry-safe unit of work.

In [ ]:
from sqlalchemy import text

with engine.begin() as connection:
    observed_database = connection.scalar(text("SELECT current_database()"))

assert observed_database == "advanced_sql_training"

### 6. Know when to leave magics

Use magics for interactive, bounded exploration. Prefer Psycopg application code for reusable functions, typed row mapping, COPY/streaming, explicit transactions, retry classification, pooling, async work, cancellation, structured logs, and fake-backed tests. A notebook can call those tested functions instead of becoming the only copy of production logic.

## Checks

Complete these without opening the solution notebook:

1. Write a query that returns US customers whose lifetime order total is at least `exercise_minimum_total`.
2. Bind both values with `:exercise_country` and `:exercise_minimum_total`.
3. Order by total descending and customer ID ascending; fetch no more than 10 rows.
4. Assign the result and convert it to a DataFrame.
5. Explain why replacing either value with `{{...}}` changes the safety boundary.

In [ ]:
exercise_country = "US"
exercise_minimum_total = 500

# Add your result assignment and bound aggregate query in the next cell.

In [ ]:
%%sql
-- Replace this diagnostic row with the aggregate query described above.
SELECT :exercise_country AS country_to_bind,
       :exercise_minimum_total AS minimum_to_bind;

Before continuing, verify that the notebook contains no URL literal or saved output; every exploratory query is bounded; values use `:name`; identifiers are static; and no write was issued. Run `%sql --connections` to inspect the active alias, then close it explicitly.

In [ ]:
%sql --close ds60-course

In [ ]:
engine.dispose()

## Next Steps

Compare behavior with the separate solution notebook only after attempting the check. Then return to Bridge Days 3–5 for production-safe Psycopg adapters and tests, or continue to BRIDGE-OPS-01 for migration delivery, readiness, and redacted observability.